### Clone the repo

In [1]:
# !git clone https://github.com/MilyaushaShamsutdinova/AlignScore

In [1]:
%cd AlignScore

/kaggle/working/AlignScore


### Load our metric model

In [2]:
# %%bash
!pip install huggingface_hub[hf_transfer]
!export HF_HUB_ENABLE_HF_TRANSFER=1

In [3]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(repo_id="CatFr0g/ruAlignScore", filename="RuAlignScore.ckpt")
path

RuAlignScore.ckpt:   0%|          | 0.00/2.50G [00:00<?, ?B/s]

'/root/.cache/huggingface/hub/models--CatFr0g--ruAlignScore/snapshots/bec0aa9d845e8ec27e9df3b598a1655c7d571636/RuAlignScore.ckpt'

In [4]:
# !pip install -e .
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 10.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing

In [5]:
from src.AlignScore import AlignScore

scorer = AlignScore(
    model='bert-base-uncased',
    batch_size=32,
    device='cuda:0',
    # device='cpu',
    ckpt_path=path,
    evaluation_mode='nli_sp'
)

2025-04-30 06:42:02.979211: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745995323.250343      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745995323.319436      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
score = scorer.score(contexts=['привет мир.'], claims=['привет мир.'])
print(score)

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  1.69it/s]

[0.9889354109764099]


### Load model for testing

In [7]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
google_api = user_secrets.get_secret("GOOGLE_API_KEY")

In [8]:
import google.generativeai as genai
from google.api_core import retry
import time


class GeminiAI:
    def __init__(self):
        genai.configure(api_key=google_api)
        self.model = genai.GenerativeModel(model_name="gemini-1.5-flash")

    def get_response(self, prompt: str):
        try:
            time.sleep(10)
            response = self.model.generate_content(prompt)
            return response.text
        except Exception as e:
            print(e)
            return None
    
    def get_summary(self, content: str):
        try:
            time.sleep(10)
            prompt = f"Суммаризируй следующий текст выделив только важные части: {content}"
            response = self.model.generate_content(prompt, request_options={'retry': retry.Retry()})
            return response.text
        except Exception as e:
            print(e)
            return None

In [9]:
gemini = GeminiAI()

content = "Одна из руководителей Координационного совета оппозиции Белоруссии (КС) Мария Колесникова и члены штаба экс-кандидата в президенты РБ Виктора Бабарико объявили о создании оппозиционной политической партии «Вместе». Об этом говорится в видеообращении, опубликованном на YouTube-канале Бабарико. «Пришло время объявить о продолжении нашей борьбы, но уже с наличием какой-то организационной формы, в которую мы могли бы объединить всех сторонников», — подчеркнул бывший кандидат на пост президента республики. В штабе белорусского КС отметили в свою очередь, что формат работы совета не изменится — он не является политической партией. Создание «Вместе» — это инициатива Бабарико. «Создание партии было согласовано с теми членами президиума КС, с которыми было возможно по объективным обстоятельствам», — подчеркнули в пресс-службе Координационного совета. Бабарико добавил, что за последние 3 месяца белорусская оппозиция смогла добиться большего, чем за все 26 лет правления президента Александра Лукашенко. Двумя днями ранее, 30 августа, член президиума КС в Белоруссии Мария Колесникова заявила, что совет отказывается от денег, выделенных на помощь белорусской оппозиции Евросоюзом. По ее словам, такое решение было принято, поскольку Координационный совет не просил о предоставлении каких-либо финансов. «Мы не собираемся иметь к этим деньгам никакого отношения, мы не собираемся их распределять», — отметила Колесникова. Она добавила, что о выделенных Брюсселем €53 млн члены президиума КС узнали из газет. Кроме того, 27 августа Координационный совет заявил, что не примет никакой материальной помощи из-за рубежа. «Мы придерживаемся позиции невмешательства во внутренние дела Беларуси, мы просим любых иностранных партнеров воздержаться от заявлений о представлении интересов белорусского общества», — говорится в заявлении совета. 31 августа член президиума КС Павел Латушко заявил, что в Белоруссии может появиться общественное движение, которое будет вести переговоры с властью. Это движение должно быть зарегистрировано в соответствии с законодательством, добавил он, а войдут в его состав представители различных частей белорусского общества. Такой шаг, добавил политик, позволит продемонстрировать властям «субъект переговоров». Бывший кандидат на президентских выборах в Белоруссии Светлана Тихановская, по официальным данным проигравшая на выборах, но результатов не признавшая, объявила о создании в республике Координационного совета оппозиции 14 августа — через неделю после голосования. После закрытия избирательных участков в республике начались массовые акции протеста против фальсификаций. Штабы оппозиционных кандидатов объявили о непризнании результатов выборов — согласно данным ЦИК РБ, победил с 80% голосов бессменный с 1994 года глава страны Александр Лукашенко. Разгоняя протестующих, белорусская милиция и внутренние войска применяли слезоточивый газ, светошумовые гранаты и резиновые пули. По сообщениям белорусских и российских журналистов с места событий, подавление протестов приобретало крайне жестокие формы, сотни человек были избиты, многие подвергались истязаниям в СИЗО. В дальнейшем в Белоруссии объявили о начале «всеобщей забастовки» на крупнейших промышленных предприятиях. Рабочие требовали проведения новых выборов и прекращения жестокостей со стороны силовиков. Замглавы министерства труда и социальной защиты Белоруссии Игорь Старовойтов заявил, что политически мотивированные забастовки «незаконны» и с помощью стачек решать нужно вопросы, входящие в компетенцию руководства предприятия. В конце августа президент РФ Владимир Путин заявил, что Москва сформировала резерв из сотрудников правоохранительных органов для «оказания помощи» Белоруссии, но не будет его задействовать, «пока экстремистские элементы не перейдут границ, не начнут разбой»."
response = gemini.get_summary(content)
print(response)

Белорусская оппозиция, включая Марию Колесникову и Виктора Бабарико, создала новую политическую партию «Вместе».  Это инициатива Бабарико, согласованная частично с членами Координационного совета оппозиции (КС), который, однако, сохранит свой формат работы и не станет партией.  КС отказался от финансовой помощи ЕС в размере €53 млн, заявив о нежелании вмешательства извне.  Параллельно обсуждается создание зарегистрированного общественного движения для ведения переговоров с властями.  Протесты в Белоруссии, начавшиеся после президентских выборов, продолжаются, включая всеобщую забастовку,  в то время как Россия держит в резерве силы правоохранительных органов на случай обострения ситуации.



In [10]:
correct_summary = "Белорусская оппозиция в лице экс-кандидата в президенты РБ Виктора Бабарико и координатора его штаба Марии Колесниковой объявила о создании политической партии «Вместе». По словам Бабарико, задержанного ранее белорусскими властями, для продолжения борьбы необходима «какая-то организационная форма», способная объединить всех сторонников."

score = scorer.score(contexts=[correct_summary], claims=[response])
print(score)

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  3.23it/s]

[0.7372362613677979]


### Load dataset

In [11]:
from datasets import load_dataset

ds = load_dataset("IlyaGusev/gazeta", split="test")

README.md:   0%|          | 0.00/13.4k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/252M [00:00<?, ?B/s]

0001.parquet:   0%|          | 0.00/22.7M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/27.8M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/30.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60964 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6369 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6793 [00:00<?, ? examples/s]

In [12]:
ds

Dataset({
    features: ['text', 'summary', 'title', 'date', 'url'],
    num_rows: 6793
})

In [13]:
ds[0]

{'text': 'На этих выходных в Берлине прошли крупные акции протеста против введенных для борьбы с коронавирусом ограничений. Демонстранты скандировали «Путин!» По словам депутата городской палаты представителей Гуннара Линдеманна («Альтернатива для Германии»), люди выкрикивали фамилию российского президента из уважения к нему. В комментарии РИА «Новости» немецкий политик отметил, что среди населения Германии Владимир Путин имеет хорошую репутацию. По его мнению, протестующие ранее пришли к российскому посольству, чтобы «привлечь внимание к условиям в Германии», надеясь, что Россия сможет оказать влияние на канцлера ФРГ Ангелу Меркель. «На мой взгляд, опасности для посольства России не возникло ни разу», — сказал депутат. Несмотря на то что протест оказался массовым, выступления носили «преимущественно мирный характер», уверен Линдеманн. По его словам, исключением стала только ситуацию у немецкого парламента. Там «несколько странных участников демонстрации попытались штурмовать бундестаг

In [14]:
ds_part =  ds.shuffle(seed=2025).select(range(200))
ds_part

Dataset({
    features: ['text', 'summary', 'title', 'date', 'url'],
    num_rows: 200
})

### Evaluation

In [15]:
from tqdm import tqdm

generated_summaries = []
scores = []

for example in tqdm(ds_part, desc="Evaluating LLM"):
    text = example["text"]
    reference_summary = example["summary"]
    
    generated = gemini.get_summary(text)    
    score = scorer.score(contexts=[reference_summary], claims=[generated])
    
    generated_summaries.append(generated)
    scores.append(score)

Evaluating LLM: 100%|██████████| 200/200 [39:37<00:00, 11.89s/it]


In [16]:
ds_part = ds_part.add_column("gen_summary", generated_summaries)
ds_part = ds_part.add_column("factual_score", scores)

Flattening the indices:   0%|          | 0/200 [00:00<?, ? examples/s]

In [17]:
import numpy as np

mean_score = np.mean(scores)
std_score  = np.std(scores)

print(f"Evaluated {len(scores)} examples")
print(f"Average factual‐consistency score: {mean_score:.4f} ± {std_score:.4f}")

Evaluated 200 examples
Average factual‐consistency score: 0.7285 ± 0.0639
